# DDPG on LunarLander (Continuous)

CSCI 6353 · Topic 37. Deep Deterministic Policy Gradient from scratch: an actor that outputs continuous actions, a critic that scores them, replay + target networks + soft updates, Gaussian exploration noise — trained on LunarLander-v3 (continuous). ~300k steps takes a few minutes on CPU.

In [ ]:
%pip install -q "gymnasium[box2d]" imageio
import numpy as np
import torch, torch.nn as nn, torch.optim as optim
import gymnasium as gym

env = gym.make("LunarLander-v3", continuous=True)
obs_dim = env.observation_space.shape[0]      # 8
act_dim = env.action_space.shape[0]           # 2, in [-1, 1]
print("obs:", obs_dim, " act:", act_dim, env.action_space.low, env.action_space.high)

## The two networks

Actor mu(s): state -> action, tanh output keeps it in [-1, 1]. Critic Q(s, a): state AND action in, one score out.

In [ ]:
class Actor(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
            nn.Linear(256, act_dim), nn.Tanh())   # actions in [-1, 1]
    def forward(self, s): return self.net(s)

class Critic(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim + act_dim, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
            nn.Linear(256, 1))
    def forward(self, s, a): return self.net(torch.cat([s, a], dim=1))

torch.manual_seed(0); np.random.seed(0)
actor, critic = Actor(), Critic()
actor_t, critic_t = Actor(), Critic()               # slow target copies
actor_t.load_state_dict(actor.state_dict()); critic_t.load_state_dict(critic.state_dict())
a_opt = optim.Adam(actor.parameters(), lr=1e-4)
c_opt = optim.Adam(critic.parameters(), lr=1e-3)

## Replay buffer and soft update

In [ ]:
GAMMA, TAU, BATCH, BUF = 0.99, 0.005, 128, 200_000
S  = np.zeros((BUF, obs_dim), np.float32); A = np.zeros((BUF, act_dim), np.float32)
R  = np.zeros((BUF, 1), np.float32); S2 = np.zeros((BUF, obs_dim), np.float32)
D  = np.zeros((BUF, 1), np.float32)
ptr = size = 0

def soft_update(net, target):                        # theta' <- 0.995 theta' + 0.005 theta
    with torch.no_grad():
        for p, pt in zip(net.parameters(), target.parameters()):
            pt.data.mul_(1 - TAU).add_(TAU * p.data)

## The training loop

Act with Gaussian noise -> store -> sample a mini-batch -> critic regresses to y = r + gamma(1-d) Q'(s', mu'(s')) -> actor ascends Q(s, mu(s)) -> soft-update both targets. The first 10k steps act at random to fill the buffer; we checkpoint the best-evaluating actor because DDPG is famously unstable.

In [ ]:
TOTAL_STEPS   = 300_000     # ~8 min on a laptop CPU (reduce to 150_000 if impatient)
START_RANDOM  = 10_000
NOISE0, NOISE_MIN = 0.3, 0.05

def evaluate(k=3):
    ev = gym.make("LunarLander-v3", continuous=True); rets = []
    for i in range(k):
        s, _ = ev.reset(seed=1000 + i); ret = 0.0
        while True:
            with torch.no_grad():
                a = actor(torch.tensor(s).unsqueeze(0)).squeeze(0).numpy()
            s, r, t1, t2, _ = ev.step(a); ret += r
            if t1 or t2: break
        rets.append(ret)
    return float(np.mean(rets))

returns, best = [], -1e9
s, _ = env.reset(seed=0); ep_ret = 0.0
for step in range(1, TOTAL_STEPS + 1):
    if step <= START_RANDOM:
        a = env.action_space.sample()
    else:
        noise = max(NOISE_MIN, NOISE0 * (1 - step / TOTAL_STEPS))
        with torch.no_grad():
            a = actor(torch.tensor(s).unsqueeze(0)).squeeze(0).numpy()
        a = np.clip(a + np.random.normal(0, noise, act_dim), -1, 1).astype(np.float32)

    s2, r, term, trunc, _ = env.step(a)
    S[ptr], A[ptr], R[ptr], S2[ptr], D[ptr] = s, a, r, s2, float(term)
    ptr = (ptr + 1) % BUF; size = min(size + 1, BUF)
    s = s2; ep_ret += r
    if term or trunc:
        returns.append(ep_ret); s, _ = env.reset(); ep_ret = 0.0

    if step > START_RANDOM and size >= BATCH:
        idx = np.random.randint(0, size, BATCH)
        bs, ba = torch.tensor(S[idx]), torch.tensor(A[idx])
        br, bs2, bd = torch.tensor(R[idx]), torch.tensor(S2[idx]), torch.tensor(D[idx])
        with torch.no_grad():                                   # the DDPG target
            y = br + GAMMA * (1 - bd) * critic_t(bs2, actor_t(bs2))
        c_loss = nn.functional.mse_loss(critic(bs, ba), y)      # critic: regression to y
        c_opt.zero_grad(); c_loss.backward(); c_opt.step()
        a_loss = -critic(bs, actor(bs)).mean()                  # actor: ascend Q(s, mu(s))
        a_opt.zero_grad(); a_loss.backward(); a_opt.step()
        soft_update(actor, actor_t); soft_update(critic, critic_t)

    if step % 20_000 == 0:
        m = evaluate()
        print(f"step {step:>7}  eval {m:7.1f}")
        if m > best:
            best = m; torch.save(actor.state_dict(), "actor_best.pt")
torch.save(actor.state_dict(), "actor_final.pt")
print("best eval:", round(best, 1))


## The learning curve

Expect the classic DDPG shape: chaotic start, a rise, often a collapse, then recovery — the instability TD3 (next topic) was built to fix.

In [ ]:
import matplotlib.pyplot as plt
rets = np.array(returns); w = 20
ma = np.convolve(rets, np.ones(w)/w, mode="valid")
plt.figure(figsize=(9, 4))
plt.plot(rets, color="#93c5fd", lw=0.8, label="episode return")
plt.plot(range(w-1, len(rets)), ma, color="#2563eb", lw=2, label=f"moving avg ({w})")
plt.axhline(200, ls="--", color="green", label="200 = solved")
plt.xlabel("episode"); plt.ylabel("return"); plt.legend(); plt.grid(alpha=.3); plt.show()

## Watch the best actor land

Reload the best checkpoint, run it deterministically (noise off), and render a GIF.

In [ ]:
import imageio
import os
ckpt = "actor_best.pt" if os.path.exists("actor_best.pt") else "actor_final.pt"
actor.load_state_dict(torch.load(ckpt)); actor.eval()
print("loaded", ckpt)

# deterministic evaluation over 20 fresh episodes
scores = {}
for seed in range(20):
    ev = gym.make("LunarLander-v3", continuous=True)
    s, _ = ev.reset(seed=seed); ret = 0.0
    while True:
        with torch.no_grad():
            a = actor(torch.tensor(s).unsqueeze(0)).squeeze(0).numpy()
        s, r, t1, t2, _ = ev.step(a); ret += r
        if t1 or t2: break
    scores[seed] = ret
print("mean %.1f  best %.1f" % (np.mean(list(scores.values())), max(scores.values())))

# render the best-scoring seed
good = max(scores, key=scores.get)
ev = gym.make("LunarLander-v3", continuous=True, render_mode="rgb_array")
s, _ = ev.reset(seed=good); frames = []
while True:
    frames.append(ev.render())
    with torch.no_grad():
        a = actor(torch.tensor(s).unsqueeze(0)).squeeze(0).numpy()
    s, r, t1, t2, _ = ev.step(a)
    if t1 or t2: break
imageio.mimsave("lander.gif", frames[::2], fps=25, loop=0)
print("saved lander.gif —", len(frames), "frames")
